In [5]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load .env file
load_dotenv(dotenv_path='../.env')

api_key = os.getenv("GROQ_API_KEY")
print("API Key loaded:", api_key is not None)

client = Groq(api_key=api_key)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",  # Free and active model
    messages=[
        {"role": "user", "content": "Say 'Connection successful' if you can read this."}
    ]
)

print(response.choices[0].message.content)

API Key loaded: True
Connection successful.


In [6]:
import pandas as pd

# Load the data pieces needed to build corridor context
corridor_stats = pd.read_csv('../data/processed/corridor_risk_tiers.csv')
trip_df = pd.read_csv('../data/processed/trip_level_cleaned.csv')

def get_corridor_context(corridor_id):
    """
    Builds a factual, real-data summary of a corridor's performance.
    This is what we feed to the LLM instead of letting it guess/hallucinate.
    """
    row = corridor_stats[corridor_stats['corridor_id'] == corridor_id]
    
    if row.empty:
        return f"No data found for corridor: {corridor_id}"
    
    row = row.iloc[0]
    
    # Route-type breakdown for this specific corridor
    corridor_trips = trip_df[trip_df['corridor_id'] == corridor_id]
    route_breakdown = corridor_trips.groupby('route_type')['delay_percentage'].mean().round(2).to_dict()
    
    context = f"""
Corridor: {corridor_id}
Risk Tier: {row['risk_tier']}
Total Trips: {row['total_trips']}
Average Delay %: {row['avg_delay_pct']:.2f}%
Delay Rate (% of trips delayed): {row['delay_rate']*100:.2f}%
Route Type Breakdown (avg delay % by type): {route_breakdown}
"""
    return context

# Test on a known Critical corridor
sample_context = get_corridor_context('Maharashtra_Maharashtra')
print(sample_context)


Corridor: Maharashtra_Maharashtra
Risk Tier: Critical
Total Trips: 2406
Average Delay %: 203.73%
Delay Rate (% of trips delayed): 98.84%
Route Type Breakdown (avg delay % by type): {'Carting': 217.11, 'FTL': 141.93}



In [9]:
def build_system_prompt():
    """
    Structured prompt using: Role, Task, Constraints, Output Format (JSON), 
    Few-Shot Example, and Fallback.
    """

    prompt = """
### ROLE ###
You are a senior logistics operations analyst working for Delhivery, India's largest logistics company. You specialize in explaining delivery delay risks to non-technical operations managers in clear business language.

### TASK ###
Given real operational data about a specific delivery corridor, explain why it is classified as high-risk (or otherwise), and provide one clear, actionable recommendation the operations team can act on immediately.

### CONSTRAINTS ###
- Use ONLY the data provided in the user message. Never invent numbers, percentages, or facts not explicitly given.
- Do not mention weather, traffic accidents, or any external cause unless it is present in the given data.
- Do not use technical ML/statistics jargon (e.g. "feature importance", "regression coefficient").
- Each field in the output must be a single, concise sentence or two. No markdown formatting (no **, no bullet points) inside the JSON values.
- Respond with VALID JSON ONLY. No text before or after the JSON. No markdown code fences (no ```json).

### OUTPUT FORMAT ###
Respond with a single JSON object in exactly this structure:

{
  "corridor_id": "<the corridor id>",
  "risk_tier": "<the risk tier from the data>",
  "root_cause": "<why this corridor has this risk level, based only on given numbers>",
  "recommended_action": "<one specific, practical action>",
  "business_impact": "<what happens if not addressed>"
}

### FEW-SHOT EXAMPLE ###
Example Input Data:
Corridor: Delhi_Haryana
Risk Tier: Watch
Total Trips: 396
Average Delay %: 129.19%
Delay Rate: 93.69%
Route Type Breakdown: {'Carting': 145.30, 'FTL': 98.20}

Example Output:
{
  "corridor_id": "Delhi_Haryana",
  "risk_tier": "Watch",
  "root_cause": "This corridor shows a moderate but consistent delay pattern, with Carting trips at 145% average delay performing notably worse than FTL trips at 98%, suggesting local handling is the main contributor.",
  "recommended_action": "Review hub loading and dispatch procedures specifically for Carting-mode shipments on this route, since FTL trips on the same corridor perform significantly better.",
  "business_impact": "If unaddressed, this corridor risks escalating from Watch to Critical tier as delay rates are already above 90%."
}

### FALLBACK ###
If the corridor data provided says "No data found" or is missing key fields, respond with this exact JSON:
{
  "corridor_id": "unknown",
  "risk_tier": "unknown",
  "root_cause": "Insufficient data available for this corridor.",
  "recommended_action": "Verify the corridor ID or check if it meets the minimum trip volume threshold.",
  "business_impact": "Cannot be assessed without sufficient data."
}
"""
    return prompt

In [10]:
import json

def ask_ai_assistant(corridor_id, user_question=None):
    """
    Sends real corridor data to the LLM and returns a structured Python dict
    (parsed from JSON), ready to use in Streamlit, dashboards, or automation.
    """
    context = get_corridor_context(corridor_id)
    
    if user_question is None:
        user_question = f"Explain the risk for corridor: {corridor_id}"

    system_prompt = build_system_prompt()

    user_prompt = f"""
Here is the real operational data for this corridor:

{context}

Question: {user_question}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3,
        response_format={"type": "json_object"}  # forces the model to return valid JSON
    )
    
    raw_output = response.choices[0].message.content
    
    # Parse into a Python dictionary for easy use downstream
    try:
        result = json.loads(raw_output)
    except json.JSONDecodeError:
        result = {"error": "Failed to parse AI response as JSON", "raw": raw_output}
    
    return result

# Test it
result = ask_ai_assistant('Maharashtra_Maharashtra')
result

{'corridor_id': 'Maharashtra_Maharashtra',
 'risk_tier': 'Critical',
 'root_cause': 'Almost all trips (98.84%) are delayed and the average delay is over 200%, with Carting trips showing the highest delay at 217.11% compared to FTL at 141.93%, indicating systemic handling issues on this corridor.',
 'recommended_action': 'Conduct an immediate audit of Carting operations on this route and reallocate capacity or improve hub processing to reduce the excessive delay.',
 'business_impact': 'If the delays persist, customer satisfaction will fall sharply, revenue will be eroded and the corridor may incur contractual penalties for missing service levels.'}

In [11]:
# Test across different risk tiers to make sure the assistant handles all cases well
test_corridors = ['Maharashtra_Maharashtra', 'Delhi_Haryana', 'Karnataka_Karnataka']

for corridor in test_corridors:
    print(f"\n{'='*60}")
    result = ask_ai_assistant(corridor)
    print(f"Corridor: {result.get('corridor_id')} | Tier: {result.get('risk_tier')}")
    print(f"Root Cause: {result.get('root_cause')}")
    print(f"Action: {result.get('recommended_action')}")
    print(f"Impact: {result.get('business_impact')}")


Corridor: Maharashtra_Maharashtra | Tier: Critical
Root Cause: The corridor has an average delay of 203.73% and 98.84% of trips are delayed, with Carting trips lagging at 217.11% versus FTL at 141.93%, indicating systemic handling issues especially for Carting shipments.
Action: Conduct an immediate audit of Carting‑mode loading, dispatch and hub processing on this route and implement corrective SOPs to reduce handling time.
Impact: If the delays persist, customer satisfaction will deteriorate, service level penalties may increase and the corridor could further damage the company’s reputation and revenue.

Corridor: Delhi_Haryana | Tier: Critical
Root Cause: The corridor has a 93.69% delay rate and an average delay of 129.19%, with Carting trips delayed at 130.74% versus FTL at 89.78%, indicating that Carting operations are the main driver of the critical risk.
Action: Conduct an immediate audit of Carting handling processes at the hub and reallocate resources or shift volume to FTL w

In [12]:
# Test the fallback behavior with a non-existent corridor
fallback_test = ask_ai_assistant('FakeState_FakeState')
print(fallback_test)

{'corridor_id': 'unknown', 'risk_tier': 'unknown', 'root_cause': 'Insufficient data available for this corridor.', 'recommended_action': 'Verify the corridor ID or check if it meets the minimum trip volume threshold.', 'business_impact': 'Cannot be assessed without sufficient data.'}


In [17]:
# Test importing from the module (simulates how Streamlit will use it later)
import sys
os.chdir('..')

from app.ai_assistant import ask_ai_assistant

result = ask_ai_assistant('Maharashtra_Maharashtra')

def display_ai_response(result):
    """Prints the AI assistant's structured response in a readable, multi-line format."""
    print(f"Corridor: {result.get('corridor_id')}")
    print(f"Risk Tier: {result.get('risk_tier')}")
    print()
    print(f" Root Cause:")
    print(f"{result.get('root_cause')}")
    print()
    print(f" Recommended Action:")
    print(f"{result.get('recommended_action')}")
    print()
    print(f"  Business Impact:")
    print(f"{result.get('business_impact')}")

# Use it
result = ask_ai_assistant('Maharashtra_Maharashtra')
display_ai_response(result)

Corridor: Maharashtra_Maharashtra
Risk Tier: Critical

 Root Cause:
The corridor has an average delay of 203.73% and a delay rate of 98.84%, with Carting trips delayed at 217.11% versus FTL at 141.93%, showing severe and almost universal lateness, especially for Carting mode.

 Recommended Action:
Deploy additional resources to streamline Carting loading and unloading processes and monitor performance daily to reduce the high Carting delay.

  Business Impact:
If not fixed, the corridor will continue to miss delivery promises, erode customer trust, and increase operational costs due to repeated re‑routing and compensation.


In [25]:
# Generate and save AI explanations for all Critical and Watch tier corridors
# This creates a ready reference file for the Power BI Action Queue and for interview demos

priority_corridors = corridor_stats[corridor_stats['risk_tier'].isin(['Critical', 'Watch'])]['corridor_id'].tolist()

ai_explanations = []

for corridor in priority_corridors[:20]:  # limit to top 20 to save API calls/time
    result = ask_ai_assistant(corridor)
    ai_explanations.append(result)

ai_explanations_df = pd.DataFrame(ai_explanations)
ai_explanations_df.to_csv('data/processed/ai_corridor_explanations.csv', index=False)

print(f"Saved {len(ai_explanations_df)} AI explanations")
ai_explanations_df.head()

Saved 20 AI explanations


,corridor_id,risk_tier,root_cause,recommended_action,business_impact
0,Andhra Pradesh_Andhra Pradesh,Watch,Almost every trip (99.73%) is delayed and the ...,Conduct an immediate audit of Carting operatio...,"If the delay pattern continues, the corridor c..."
1,Andhra Pradesh_Telangana,Watch,"All 22 trips on this corridor are delayed, giv...",Conduct a quick audit of FTL loading and dispa...,"If not fixed, the corridor may see rising cust..."
2,Assam_Assam,Critical,The corridor has an extremely high average del...,Conduct an immediate audit of loading and disp...,"If not fixed, the corridor will continue to mi..."
3,Assam_Delhi,Watch,"All 17 trips on this corridor are delayed, wit...","Conduct an immediate audit of the FTL loading,...","If not fixed, the corridor will continue to mi..."
4,Assam_Maharashtra,Watch,"All 5 trips on this corridor are delayed, with...","Conduct an immediate audit of FTL loading, dis...","If not fixed, the corridor will continue to mi..."
